Generating arrays for h5 file

In [1]:
import glob
import h5py
import importlib
import IPython.display as ipd
import soxr
import numpy as np
import os
import pandas as pd
import pickle
import soundfile as sf
import src.audio_transforms as at
import src.custom_modules as cm
import src.binaural_attn_lightning as binaural_lightning
importlib.reload(binaural_lightning)
import sys
import torch
import tqdm
import yaml

from pathlib import Path
from pytorch_lightning import Trainer
from scipy import signal
from scipy.io.wavfile import read, write

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [2]:
# path to 376 wiki words
swc_path = '/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/foreground_swc/'

# path o common voice words not included in training
df = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/commonvoice_9_en/manifest_all_word_alignments.pdpkl')

In [3]:
manifest = pd.read_pickle(swc_path + 'manifest.pdpkl')

In [4]:
manifest

,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,raw_clip_dur_in_s,raw_clip_end_in_s,raw_clip_start_in_s,raw_src_fn,raw_total_file_duration_in_s,split,split_int,sr,src_fn,total_file_duration_in_s,word
0,a-c-norman,3.0,3.0,0.0,swc,0.32,2094.94,2094.62,/scratch2/weka/mcdermott/msaddler/swc/english/...,2175.444172,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,above
1,jebjoya,3.0,3.0,0.0,swc,0.49,1715.87,1715.38,/scratch2/weka/mcdermott/msaddler/swc/english/...,2793.356190,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,according
2,caninedoubletake,3.0,3.0,0.0,swc,0.36,169.03,168.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,987.438730,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,across
3,karltalk,3.0,3.0,0.0,swc,0.60,2429.51,2428.91,/scratch2/weka/mcdermott/msaddler/swc/english/...,4802.892336,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,action
4,s-whistler,3.0,3.0,0.0,swc,0.80,1149.87,1149.07,/scratch2/weka/mcdermott/msaddler/swc/english/...,4463.715556,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,activities
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371,incledon,3.0,3.0,0.0,swc,0.69,231.72,231.03,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.573152,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,world
372,ama1016,3.0,3.0,0.0,swc,0.42,193.47,193.05,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.502041,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,writing
373,zanimum,3.0,3.0,0.0,swc,0.27,13.92,13.65,/scratch2/weka/mcdermott/msaddler/swc/english/...,452.154921,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,written
374,tonyle,3.0,3.0,0.0,swc,0.31,1208.98,1208.67,/scratch2/weka/mcdermott/msaddler/swc/english/...,2359.540680,eval,2,44100,/om2/user/msaddler/spatial_audio_pipeline/asse...,3.0,wrote


In [5]:
fn_pkl_src = '/scratch2/weka/mcdermott/raygon/projects/public/jsinDataset/assets/data/interim/swc/mungedFinalDataframeWords_swc_readerNormalized_sexNormalized_accent.pdpkl'
fn_pkl_dst = '/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl'

In [6]:
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))
words = list(word_dict.keys())
words = [word.replace("'", "") for word in words]
manifest_all_words = pd.read_pickle(fn_pkl_dst)
# filter out words not in 'words' list
manifest_all_words = manifest_all_words[manifest_all_words['word'].isin(words)]
word_counts = manifest_all_words['word'].value_counts()

In [7]:
down_df = manifest_all_words.groupby(['word', 'gender']).sample(1, replace=False)

In [8]:
final_df = down_df.groupby('word').filter(lambda x: len(x) == 2)

In [9]:
final_df = final_df.reset_index().rename(columns={'index':'src_ix'})

In [10]:
final_df

,src_ix,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,sr,src_fn,total_file_duration_in_s,word
0,600986,laura-s,0.31,722.16,721.85,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1796.176689,about
1,431547,matthewdgonzalez,0.25,4753.21,4752.96,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,4801.261134,about
2,110091,popularoutcast,0.24,379.15,378.91,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,906.763900,above
3,349283,mangst,0.36,1306.60,1306.24,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3352.103764,above
4,380993,laurahale,0.37,269.10,268.73,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,371.816780,access
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1573,653426,donbert,0.35,1554.46,1554.11,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1597.876825,yellow
1574,136224,popularoutcast,0.60,780.23,779.63,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1160.012336,young
1575,631385,chadley99,0.30,822.48,822.18,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1834.808889,young
1576,703467,flyingtoaster,0.41,435.43,435.02,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3906.204444,younger


In [11]:
all_words_not_filtered = pd.read_pickle(fn_pkl_dst)
oov_words = all_words_not_filtered[~all_words_not_filtered['word'].isin(words)]

In [12]:
talkers = final_df['client_id'].unique()
oov_words[oov_words['client_id'].isin(talkers)].client_id.value_counts()

s-whistler               60197
matthewdgonzalez         49291
mangst                   36135
the-voice-of-hassocks    35187
alexkillby               26728
                         ...  
acerperi                    62
abby83                      58
cmat                        56
do-neil                     55
jake-wasdin                 48
Name: client_id, Length: 220, dtype: int64

In [13]:
samples_per_talker = {talker:count for talker,count in final_df.client_id.value_counts().items()}
viables_cues = oov_words[oov_words.client_id.isin(talkers)]

cues = viables_cues.groupby('client_id').apply(lambda group: group.sample(samples_per_talker[group.name]))
cues.drop(columns='client_id', inplace=True)
cues = cues.reset_index()
cues.rename(columns={'level_1':'cue_src_ix'}, inplace=True)

In [14]:
final_df.sort_values(by='client_id', inplace=True)
final_df.reset_index(inplace=True, drop=True)


cues.sort_values(by='client_id', inplace=True)
cues.reset_index(inplace=True, drop=True)



### Merge cues with foregrounds  
cues[['cue_word', 'cue_src_ix', 'cue_client_id', 'cue_fn', 'cue_start_in_s', 'cue_end_in_s']] = cues[['word', 'cue_src_ix', 'client_id', 'src_fn', 'clip_start_in_s', 'clip_end_in_s']]
# Combine as experiment dataframe
training_df = final_df.join(cues[['cue_word', 'cue_src_ix', 'cue_client_id', 'cue_fn', 'cue_start_in_s', 'cue_end_in_s']])
assert (training_df['client_id'] == training_df['cue_client_id']).all(), "Cue and Target talkers don't match!"

In [15]:
training_df

,src_ix,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,sr,src_fn,total_file_duration_in_s,word,cue_word,cue_src_ix,cue_client_id,cue_fn,cue_start_in_s,cue_end_in_s
0,686999,0x0077be,0.48,952.07,951.59,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,961.569333,despite,weather,684187,0x0077be,/scratch2/weka/mcdermott/msaddler/swc/english/...,361.81,362.11
1,683806,0x0077be,0.25,81.51,81.26,swc,male,1,NaN,0,48000,/scratch2/weka/mcdermott/msaddler/swc/english/...,775.033333,although,inland,684876,0x0077be,/scratch2/weka/mcdermott/msaddler/swc/english/...,244.99,245.41
2,992807,1904-cc,2.15,1369.10,1366.95,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2207.180045,community,accessible,992649,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1018.12,1018.73
3,992744,1904-cc,0.40,1146.85,1146.45,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2207.180045,software,hardware,992905,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1863.72,1864.12
4,992815,1904-cc,0.58,1378.71,1378.13,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2207.180045,produced,content,992842,1904-cc,/scratch2/weka/mcdermott/msaddler/swc/english/...,1455.50,1456.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1573,105125,zanimum,0.39,339.15,338.76,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,617.248798,forest,twelve,98644,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,335.78,336.32
1574,99531,zanimum,0.45,135.71,135.26,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,777.806077,national,scholarship,96805,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,106.35,107.02
1575,96822,zanimum,0.84,125.34,124.50,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,315.673832,california,newer,102317,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,943.46,944.14
1576,95436,zanimum,0.46,404.40,403.94,swc,male,1,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,955.601270,robert,name,104815,zanimum,/scratch2/weka/mcdermott/msaddler/swc/english/...,221.27,221.65


In [16]:
training_df = pd.read_pickle('/om2/user/rphess/Auditory-Attention/binaural_test_manifest.pdpkl')

Trial run of Testing (if this works, will spatialize the manifest and write test script)

In [17]:
def pad_or_trim_to_len(x, n, mode='both', kwargs_pad={}):
    """
    Increases or decreases the length of a one-dimensional signal
    by either padding or triming the array. If the difference
    between `len(x)` and `n` is odd, this function will default to
    adding/removing the extra sample at the end of the signal.
    
    Args
    ----
    x (np.ndarray): one-dimensional input signal
    n (int): length of output signal
    mode (str): specify which end of signal to modify
        (default behavior is to symmetrically modify both ends)
    kwargs_pad (dict): keyword arguments for np.pad function
    
    Returns
    -------
    x_out (np.ndarray): one-dimensional signal with length `n`
    """
    assert len(np.array(x).shape) == 1, "input must be 1D array"
    assert mode.lower() in ['both', 'start', 'end']
    n_diff = np.abs(len(x) - n)
    if len(x) > n:
        if mode.lower() == 'end':
            x_out = x[:n]
        elif mode.lower() == 'start':
            x_out = x[-n:]
        else:
            x_out = x[int(np.floor(n_diff / 2)):-int(np.ceil(n_diff / 2))]
    elif len(x) < n:
        if mode.lower() == 'end':
            pad_width = [0, n_diff]
        elif mode.lower() == 'start':
            pad_width = [n_diff, 0]
        else:
            pad_width = [int(np.floor(n_diff / 2)), int(np.ceil(n_diff / 2))]
        kwargs = {'mode': 'constant'}
        kwargs.update(kwargs_pad)
        x_out = np.pad(x, pad_width, **kwargs)
    else:
        x_out = x
    assert len(x_out) == n
    return x_out

In [18]:
def get_excerpt(dfi, dur=3.0, sr=50000, pad_with_context=True, jitter_fraction=0):
    """
    This function loads an audio file and excerpts a clip with the specified
    duration. Target durations that exceed clip boundaries are handled with
    zero-padding (applied to all signals but sliced away when not needed).
    This function also handles resampling (via soxr) and re-scaling.
    """
    jitter_in_s = 0
    jitter_via_zero_padding = True
    if dfi.clip_dur_in_s > dur:
        # Take a random segment if clip duration is longer than excerpt
        clip_start_in_s = np.random.uniform(
            low=dfi.clip_start_in_s,
            high=dfi.clip_start_in_s + dfi.clip_dur_in_s - dur,
            size=None)
        clip_end_in_s = clip_start_in_s + dur
        jitter_via_zero_padding = False
    else:
        # Temporally jitter clip by extending either start or end time
        jitter_in_s = np.random.uniform(
            low=-dfi.clip_dur_in_s * jitter_fraction,
            high=dfi.clip_dur_in_s * jitter_fraction,
            size=None)
        if pad_with_context:
            # If using context, adjust clip start and end times to account for jitter and context
            if jitter_in_s > 0:
                clip_start_in_s = dfi.clip_start_in_s - (2 * np.abs(jitter_in_s))
                clip_end_in_s = dfi.clip_end_in_s
            else:
                clip_start_in_s = dfi.clip_start_in_s
                clip_end_in_s = dfi.clip_end_in_s + (2 * np.abs(jitter_in_s))
            clip_dur_in_s = clip_end_in_s - clip_start_in_s
            jitter_via_zero_padding = False
            context_pad_in_s = (dur - clip_dur_in_s) / 2
        else:
            clip_start_in_s = dfi.clip_start_in_s
            clip_end_in_s = dfi.clip_end_in_s
            context_pad_in_s = 0
        clip_start_in_s = clip_start_in_s - context_pad_in_s
        clip_end_in_s = clip_end_in_s + context_pad_in_s
    clip_dur_in_s = clip_end_in_s - clip_start_in_s
    # Load audio, pad, slice with indexes that account for padding
    load_full_file = True
    if (clip_start_in_s >= 0) and (clip_end_in_s < dfi.total_file_duration_in_s):
        # Attempt to read only the specified excerpt
        myfile = sf.SoundFile(dfi.src_fn)
        if myfile.seekable():
            src_sr = myfile.samplerate
            frame_start = int(np.round(clip_start_in_s * src_sr))
            frames = int(np.round(clip_dur_in_s * src_sr))
            myfile.seek(frame_start)
            y = myfile.read(frames, always_2d=True)
            y = np.mean(y, axis=1)
            load_full_file = False
    if load_full_file:
        # If impossible, read full audio file
        y, src_sr = sf.read(dfi.src_fn, always_2d=True)
        y = np.mean(y, axis=1)
        frame_start = int(np.round(clip_start_in_s * src_sr))
        frames = int(np.round(clip_dur_in_s * src_sr))
        if frame_start < 0:
            y = np.pad(y, [-frame_start, 0])
            frame_start = 0
        if frame_start + frames > len(y):
            y = np.pad(y, [0, frame_start + frames - len(y)])
        y = y[frame_start : frame_start + frames]
    # Resample from src_sr to sr
    y = soxr.resample(y, src_sr, sr).astype(np.float32)
    # If not yet jittered, apply jitter at end via asymmetric zero-padding
    if jitter_via_zero_padding:
        jitter_pad_width = int(np.round(2 * np.abs(jitter_in_s) * sr))
        if jitter_in_s > 0:
            y = np.pad(y, [jitter_pad_width, 0]).astype(np.float32)
        else:
            y = np.pad(y, [0, jitter_pad_width]).astype(np.float32)
    # Zero-pad or trim to length (fixes off by one errors)
    y = pad_or_trim_to_len(y, int(dur * sr))
    y = np.nan_to_num(y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return y

In [19]:
df = training_df.sample(350)

In [20]:
foregrounds = []
cues = []
for i, row in df.iterrows():
    src_ix = row['src_ix']
    cue_ix = row['cue_src_ix']
    src_row = all_words_not_filtered.iloc[src_ix]
    cue_row = all_words_not_filtered.iloc[cue_ix]
    foregrounds.append(get_excerpt(src_row))
    cues.append(get_excerpt(cue_row))

In [21]:
df['loaded_foreground'] = foregrounds

In [22]:
df['loaded_cue'] = cues

In [23]:
mdl_ckpt = 'attn_cue_models/binaural_word_task_cue_voiec_and_loc_v02/checkpoints/epoch=0-step=2000-v3.ckpt'

In [24]:
config = yaml.load(open('/om2/user/imgriff/projects/Auditory-Attention/config/binaural_attn/dev_voice_and_loc_cue_001.yaml', 'r'), Loader=yaml.FullLoader)

In [25]:
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0

In [26]:
model = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=mdl_ckpt, config=config).cuda()

num_classes={'num_words': 800}
Model performing word task
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


Case 1: target at 0, distractor at 90
Case 2: target at -90, distractor at 90
Case 3: both at 0

In [27]:
def mass_spatialize(words, ir):
    """Uses pytorch to convolve all sounds in words with 2 channel IR given in ir"""
    n_words = words.shape[0]
    words_padded = [torch.nn.functional.pad(word, (ir.shape[0] - 1, 0)) for word in words]
    ir = ir.T.unsqueeze(1)
    words_padded = torch.stack(words_padded)
    spatialized = torch.nn.functional.conv1d(words_padded.view(n_words, 1, -1).cuda(), ir.cuda()).cuda()
    return spatialized

In [28]:
print("Loading speaker array room BRIRs")
list_data_dict = []
for elev in [-20, -10, 0, 10, 20, 30, 40]:
    for azim in np.arange(0, 360, 5):
        data_dict = {
            'azim': azim,
            'elev': elev,
            'brir': [],
        }
        for ear in ['l', 'r']:
            basename = f'{elev}elev_{azim}az_2.47x2.60y2.00z_{ear}.wav'
            if elev >= 0:
                fn = os.path.join('/om/user/francl/Room_Simulator_20181115_Rebuild/room_HRIRs/', basename)
            else:
                fn = os.path.join('/om/user/francl/Room_Simulator_20181115_Rebuild/room_HRIRs/neg_elevs/', basename)
            assert os.path.exists(fn)
            brir, sr_src = sf.read(fn)
            sr = 50000
            brir = soxr.resample(brir.astype(np.float32), sr_src, sr)
            data_dict['brir'].append(brir)
        data_dict['sr'] = sr
        data_dict['brir'] = np.stack(data_dict['brir']).T
        list_data_dict.append(data_dict)
df_brir = pd.DataFrame(list_data_dict)
df_brir_room = df_brir[np.logical_and.reduce([
    df_brir['azim'] % 10 == 0,
    ~(np.logical_and(df_brir['azim'] > 90, df_brir['azim'] < 270)),
    df_brir['elev'] >= 0,
])].reset_index()
print(f"Loaded speaker array room BRIRs ({len(df_brir_room)})")

Loading speaker array room BRIRs
Loaded speaker array room BRIRs (95)


In [29]:
brir_00 = torch.from_numpy(df_brir_room[(df_brir_room['azim'] == 0) & (df_brir_room['elev'] == 0)]['brir'].values[0])
brir_900 = torch.from_numpy(df_brir_room[(df_brir_room['azim'] == 270) & (df_brir_room['elev'] == 0)]['brir'].values[0])
brir_neg900 = torch.from_numpy(df_brir_room[(df_brir_room['azim'] == 90) & (df_brir_room['elev'] == 0)]['brir'].values[0])

In [30]:
brir_00.shape

torch.Size([25000, 2])

In [31]:
brir_00 = torch.flip(brir_00, dims=[0])
brir_900 = torch.flip(brir_900, dims=[0])
brir_neg900 = torch.flip(brir_neg900, dims=[0])

In [32]:
audio_transforms = model.audio_transforms

In [33]:
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
class_map = {k.replace("'", ''):v for k,v in class_map.items()}

In [34]:
model.config['num_workers'] = 10
model.config['hparas']['batch_size'] = 32

In [38]:
# dataloader = model.val_dataloader()

49 files in val concat dataset


In [39]:
# # testing with the model val loader
# results = []
# for i, batch in tqdm.tqdm(enumerate(dataloader), total=100):
#     cue, cue_mask_ix, scene, label = batch

#     out = model.forward(cue.cuda(), scene.cuda(), None)
#     softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
#     result = torch.argmax(softmax_outputs, dim=-1).cpu()
#     results.append(label == result)
#     if i == 100:
#         break
# result_tensor = torch.concat(results)
# torch.mean(result_tensor)


100%|██████████| 100/100 [04:32<00:00,  2.73s/it]


tensor(0.3320)

In [42]:
rev_map = {v:k for k,v in class_map.items()}

In [43]:
# rev_map[result]

In [42]:
model = model.eval()

In [43]:
# case 1 target at 0, distractor at right 90 

results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    fg = mass_spatialize(fg.cuda(), brir_00.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")


100%|██████████| 350/350 [02:06<00:00,  2.77it/s]

Accuracy = 0.34
Confusions = 0.017142857142857144


In [44]:
# case 2 - target left 90 distractor right 90 

results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_neg900.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    fg = mass_spatialize(fg.cuda(), brir_neg900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

  0%|          | 0/350 [00:00<?, ?it/s]

100%|██████████| 350/350 [02:19<00:00,  2.52it/s]

Accuracy = 0.58
Confusions = 0.0


In [45]:
# target at right 90, distractor at 0
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_900.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    fg = mass_spatialize(fg.cuda(), brir_900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_00.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

  0%|          | 0/350 [00:00<?, ?it/s]

  9%|▊         | 30/350 [00:10<02:27,  2.16it/s]

In [ ]:
# see if we can put cue at different location 
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_00.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    fg = mass_spatialize(fg.cuda(), brir_900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_neg900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

100%|██████████| 350/350 [02:09<00:00,  2.70it/s]

Accuracy = 0.09714285714285714
Accuracy = 0.014285714285714285


In [35]:
# Put cue at distractor location 
results = []
confusions = []
for i, row in tqdm.tqdm(df.iterrows(), total = 350):
    cue = torch.from_numpy(row['loaded_cue']).unsqueeze(0)
    fg = torch.from_numpy(row['loaded_foreground']).unsqueeze(0)
    client_id = row['client_id']
    word = row['word']
    label = class_map[word]
    distractor = df[df['client_id'] != client_id].sample(1)
    distractor_signal = torch.from_numpy(distractor['loaded_foreground'].values[0]).unsqueeze(0)
    distractor_label = class_map[distractor['word'].values[0]]
    # distractor = torch.from_numpy(df[df['client_id'] != client_id]['loaded_foreground'].sample(1).values[0]).unsqueeze(0)

    cue = mass_spatialize(cue.cuda(), brir_neg900.cuda()).cpu()
    cue = np.array(cue[:, :, 12500:137500])
    cue = audio_transforms(cue, None)[0].squeeze(0)

    fg = mass_spatialize(fg.cuda(), brir_900.cuda()).cpu()
    fg = np.array(fg[:, :, 12500:137500])
    bg = mass_spatialize(distractor_signal.cuda(), brir_neg900.cuda()).cpu()
    bg = np.array(bg[:, :, 12500:137500])
    scene = audio_transforms(fg, bg)[0].squeeze(0)

    out = model.forward(cue.cuda(), scene.cuda(), None)
    softmax_outputs = torch.nn.functional.softmax(out, dim=-1)
    result = int(torch.argmax(softmax_outputs, dim=-1).cpu())
    results.append(label == result)
    confusions.append(distractor_label == result)
print(f"Accuracy = {np.mean(results)}")
print(f"Confusions = {np.mean(confusions)}")

  0%|          | 0/350 [00:00<?, ?it/s]

100%|██████████| 350/350 [01:00<00:00,  5.74it/s]

Accuracy = 0.0
Confusions = 0.4


In [37]:
torch.Tensor([1, 2, 3]).max(-1)

torch.return_types.max(
values=tensor(3.),
indices=tensor(2))